# Signal Processing with SciRS2

This notebook demonstrates digital signal processing using `scirs2-signal`
and `scirs2-fft`, which mirror `scipy.signal` and `scipy.fft`.

Topics covered:
- Generating test signals
- FFT and frequency-domain analysis
- Butterworth filter design
- Applying filters with `filtfilt`
- Convolution
- Spectral analysis with periodogram and Welch's method

In [ ]:
:dep scirs2-core = { path = "../scirs2-core" }
:dep scirs2-fft = { path = "../scirs2-fft" }
:dep scirs2-signal = { path = "../scirs2-signal" }

## Generating a Test Signal

We create a signal composed of two sinusoids at 50 Hz and 200 Hz,
sampled at 1000 Hz. This is a common setup for testing frequency
analysis and filtering.

In [ ]:
use std::f64::consts::PI;

let fs = 1000.0;  // Sampling frequency (Hz)
let n = 1024;      // Number of samples
let dt = 1.0 / fs;

// Time vector
let t: Vec<f64> = (0..n).map(|i| i as f64 * dt).collect();

// Signal: 50 Hz sine (amplitude 1.0) + 200 Hz sine (amplitude 0.5)
let signal: Vec<f64> = t.iter()
    .map(|&ti| (2.0 * PI * 50.0 * ti).sin() + 0.5 * (2.0 * PI * 200.0 * ti).sin())
    .collect();

println!("Signal length: {} samples", signal.len());
println!("Duration: {:.3} seconds", n as f64 * dt);
println!("First 10 samples: {:?}", &signal[..10]);

## FFT: Time to Frequency Domain

The Fast Fourier Transform converts our time-domain signal into
its frequency-domain representation. We use `rfft` for real-valued
signals, which is more efficient than the full complex FFT.

In [ ]:
use scirs2_fft::rfft;

// Compute the real FFT
let spectrum = rfft(&signal, None).expect("rfft failed");

// Compute magnitude spectrum
let magnitudes: Vec<f64> = spectrum.iter()
    .map(|c| (c.re * c.re + c.im * c.im).sqrt() * 2.0 / n as f64)
    .collect();

// Frequency bins
let freq_resolution = fs / n as f64;

// Find peaks (frequencies with significant magnitude)
println!("Frequency resolution: {:.2} Hz", freq_resolution);
println!("\nDominant frequencies:");
for (i, &mag) in magnitudes.iter().enumerate() {
    if mag > 0.3 {
        println!("  {:.1} Hz: amplitude = {:.4}", i as f64 * freq_resolution, mag);
    }
}

## Inverse FFT

We can recover the original signal from the frequency domain using `irfft`.

In [ ]:
use scirs2_fft::irfft;

// Inverse real FFT to recover the time-domain signal
let recovered = irfft(&spectrum, Some(n)).expect("irfft failed");

// Check reconstruction error
let max_error = signal.iter().zip(recovered.iter())
    .map(|(a, b)| (a - b).abs())
    .fold(0.0_f64, f64::max);
println!("Max reconstruction error: {:.2e}", max_error);

## Butterworth Filter Design

We design a lowpass Butterworth filter to remove the 200 Hz component
while keeping the 50 Hz signal. The cutoff is set at 100 Hz.

The `butter` function returns numerator (b) and denominator (a)
coefficients for the IIR filter.

In [ ]:
use scirs2_signal::{butter, FilterType};

// Design a 4th-order Butterworth lowpass filter
// Cutoff at 100 Hz, normalized to Nyquist (fs/2 = 500 Hz)
let cutoff = 100.0 / (fs / 2.0);  // Normalized cutoff
let order = 4;

let (b, a) = butter(order, cutoff, FilterType::Lowpass, false)
    .expect("butter failed");

println!("Filter order: {}", order);
println!("Normalized cutoff: {:.4}", cutoff);
println!("Numerator (b): {:?}", &b[..b.len().min(6)]);
println!("Denominator (a): {:?}", &a[..a.len().min(6)]);

## Applying the Filter

We use `filtfilt` for zero-phase filtering. This applies the filter
forward and then backward, eliminating phase distortion.

In [ ]:
use scirs2_signal::filtfilt;

// Apply the Butterworth filter with zero-phase filtering
let filtered = filtfilt(&b, &a, &signal).expect("filtfilt failed");

println!("Original signal first 10 samples:  {:?}",
    signal[..10].iter().map(|x| format!("{:.4}", x)).collect::<Vec<_>>());
println!("Filtered signal first 10 samples:  {:?}",
    filtered[..10].iter().map(|x| format!("{:.4}", x)).collect::<Vec<_>>());

// Verify: the filtered signal should mostly contain the 50 Hz component
let filtered_spectrum = rfft(&filtered, None).expect("rfft failed");
let filtered_magnitudes: Vec<f64> = filtered_spectrum.iter()
    .map(|c| (c.re * c.re + c.im * c.im).sqrt() * 2.0 / n as f64)
    .collect();

println!("\nFiltered spectrum peaks:");
for (i, &mag) in filtered_magnitudes.iter().enumerate() {
    if mag > 0.1 {
        println!("  {:.1} Hz: amplitude = {:.4}", i as f64 * freq_resolution, mag);
    }
}
// The 200 Hz component should be heavily attenuated

## Convolution

Convolution is a core signal processing operation used for filtering,
smoothing, and correlation. SciRS2 provides SIMD-accelerated convolution.

In [ ]:
use scirs2_signal::convolve;

// Simple moving average (box filter) over 5 points
let data = vec![1.0, 3.0, 5.0, 7.0, 9.0, 7.0, 5.0, 3.0, 1.0];
let kernel = vec![0.2, 0.2, 0.2, 0.2, 0.2];  // 5-point average

let smoothed = convolve(&data, &kernel, "same").expect("convolve failed");
println!("Input:    {:?}", data);
println!("Smoothed: {:?}",
    smoothed.iter().map(|x| format!("{:.2}", x)).collect::<Vec<_>>());

## Spectral Analysis

The `periodogram` function estimates the power spectral density (PSD)
of a signal. `welch` provides a smoother estimate by averaging over
overlapping segments.

In [ ]:
use scirs2_signal::{periodogram, welch};

// Compute periodogram PSD estimate
let (freqs, psd) = periodogram(&signal, fs, "hann", None)
    .expect("periodogram failed");

// Find the top 3 frequency peaks
let mut indexed: Vec<(usize, f64)> = psd.iter().enumerate()
    .map(|(i, &p)| (i, p))
    .collect();
indexed.sort_by(|a, b| b.1.partial_cmp(&a.1).unwrap_or(std::cmp::Ordering::Equal));

println!("Top frequency peaks (periodogram):");
for &(i, power) in indexed.iter().take(3) {
    if i < freqs.len() {
        println!("  {:.1} Hz: PSD = {:.6}", freqs[i], power);
    }
}

In [ ]:
// Welch's method for smoother PSD estimate
let (freqs_w, psd_w) = welch(&signal, fs, "hann", Some(256), Some(128), None)
    .expect("welch failed");

let mut indexed_w: Vec<(usize, f64)> = psd_w.iter().enumerate()
    .map(|(i, &p)| (i, p))
    .collect();
indexed_w.sort_by(|a, b| b.1.partial_cmp(&a.1).unwrap_or(std::cmp::Ordering::Equal));

println!("Top frequency peaks (Welch):");
for &(i, power) in indexed_w.iter().take(3) {
    if i < freqs_w.len() {
        println!("  {:.1} Hz: PSD = {:.6}", freqs_w[i], power);
    }
}

## Summary

This notebook demonstrated the core signal processing workflow in SciRS2:

1. **Signal generation** using standard Rust math
2. **FFT analysis** with `rfft` / `irfft` for frequency-domain inspection
3. **Filter design** with `butter` (Butterworth IIR filter)
4. **Filter application** with `filtfilt` (zero-phase filtering)
5. **Convolution** with `convolve` for smoothing and FIR filtering
6. **Spectral estimation** with `periodogram` and `welch`

Additional capabilities not covered here include:
- Wavelet transforms (DWT, CWT) via `scirs2_signal::dwt` and `wavelets`
- Adaptive filters (LMS, NLMS, RLS) via `scirs2_signal::adaptive`
- Short-Time Fourier Transform via `scirs2_fft::stft`
- Beamforming via `scirs2_signal::beamforming`